# Comprehensive Research Evaluation & Ablation Study
**Nepali-English Code-Mixed Whisper ASR**

This notebook replicates the core experiments for the ACL 2023 paper:
1. **Unconstrained vs Constrained Decoding Ablation**: Proves the byte-identical output claim.
2. **Table 4 Generation**: Computes Overall WER, CER, Nep-WER, and CM-WER for the fine-tuned model and zero-shot baseline.


In [ ]:
!pip install -q transformers datasets accelerate jiwer librosa soundfile matplotlib pandas numpy seaborn tqdm scipy peft --upgrade torchao

In [ ]:
import os
import torch
import pandas as pd
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel, PeftConfig

class Config:
    base_model_id = "openai/whisper-large-v3"
    lora_model_path = "/kaggle/input/models/leo17messi/nepalienglish-codemix-model/pytorch/default/1/outputs/best_checkpoint"
    csv_path = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle2.csv"
    audio_dir = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment"
    
    test_set_size = 150
    batch_size = 8
    max_generation_length = 225

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
import re
import unicodedata
import jiwer
from collections import Counter
from tqdm import tqdm
import librosa
import time

DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F

def is_nepali(word: str) -> bool:
    return any(DEVANAGARI_START <= ord(ch) <= DEVANAGARI_END for ch in word)

def is_english(word: str) -> bool:
    try:
        word.encode('ascii')
        return any(c.isalpha() for c in word)
    except UnicodeEncodeError:
        return False

def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFC', text).strip()
    return re.sub(r'\s+', ' ', text)

def language_filtered_wer(ref: str, hyp: str, lang_func) -> float:
    ref_f = ' '.join(w for w in ref.split() if lang_func(w))
    hyp_f = ' '.join(w for w in hyp.split() if lang_func(w))
    if not ref_f.strip():
        return float('inf')
    try:
        return jiwer.wer(ref_f, hyp_f)
    except Exception:
        return float('inf')

def evaluate_predictions(references, predictions):
    wers, cers, nep_wers, cm_wers = [], [], [], []
    for ref, hyp in zip(references, predictions):
        ref_n, hyp_n = normalize_text(ref), normalize_text(hyp)
        if not ref_n.strip(): continue
        
        try:
            wers.append(jiwer.wer(ref_n, hyp_n))
            cers.append(jiwer.cer(ref_n, hyp_n))
        except:
            pass
            
        nep_wer = language_filtered_wer(ref_n, hyp_n, is_nepali)
        if nep_wer != float('inf'): nep_wers.append(nep_wer)
            
        cm_wer = language_filtered_wer(ref_n, hyp_n, is_english)
        if cm_wer != float('inf'): cm_wers.append(cm_wer)
        
    return {
        "WER": sum(wers)/len(wers) * 100 if wers else 0,
        "CER": sum(cers)/len(cers) * 100 if cers else 0,
        "Nep": sum(nep_wers)/len(nep_wers) * 100 if nep_wers else 0,
        "CM-WER": sum(cm_wers)/len(cm_wers) * 100 if cm_wers else 0
    }


In [ ]:
import csv
print('Loading hold-out test set...')
paths, texts = [], []
with open(config.csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if len(row) > 1:
            paths.append(row[0].strip())
            texts.append(','.join(row[1:]).strip())

df = pd.DataFrame({'audio_path': paths, 'text': texts})
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'] != ''].reset_index(drop=True)
df['audio_path'] = df['audio_path'].apply(
    lambda x: x if os.path.isabs(x) else os.path.join(config.audio_dir, os.path.basename(x))
)

mask = df['audio_path'].apply(os.path.exists)
df = df[mask].reset_index(drop=True)

test_df = df.iloc[-config.test_set_size:].reset_index(drop=True)
print(f'Test set ready: {len(test_df)} utterances.')


In [ ]:
def run_inference(model, processor, df, force_nepali=False):
    predictions = []
    references = df['text'].tolist()
    audio_paths = df['audio_path'].tolist()
    
    generate_kwargs = {'max_new_tokens': config.max_generation_length}
    if force_nepali:
        forced_ids = processor.get_decoder_prompt_ids(language="ne", task="transcribe")
        generate_kwargs['forced_decoder_ids'] = forced_ids
    else:
        forced_ids = processor.get_decoder_prompt_ids(task="transcribe")
        generate_kwargs['forced_decoder_ids'] = forced_ids

    for i in tqdm(range(0, len(audio_paths), config.batch_size), desc=f'Inference (Constrained={force_nepali})'):
        batch_paths = audio_paths[i:i + config.batch_size]
        batch_arrays = []
        for path in batch_paths:
            arr, sr = librosa.load(path, sr=16000, mono=True)
            batch_arrays.append(arr)

        inputs = processor(batch_arrays, sampling_rate=16000, return_tensors='pt', padding=True).to(device)
        if device.type == 'cuda':
            inputs['input_features'] = inputs['input_features'].half()

        with torch.no_grad():
            gen_ids = model.generate(inputs['input_features'], **generate_kwargs)

        preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
        predictions.extend([normalize_text(p) for p in preds])

    return predictions, references


## Run Experiments (Zero-shot vs Fine-Tuned)

In [ ]:
# 1. Load Zero-Shot Baseline Model
print("Loading Zero-Shot Baseline...")
processor = WhisperProcessor.from_pretrained(config.base_model_id)
baseline_model = WhisperForConditionalGeneration.from_pretrained(config.base_model_id).to(device)
if device.type == 'cuda': baseline_model = baseline_model.half()
baseline_model.eval()

# Run Zero-Shot Baseline (Constrained to Nepali, as per paper)
base_preds, refs = run_inference(baseline_model, processor, test_df, force_nepali=True)
base_metrics = evaluate_predictions(refs, base_preds)

# Clean up baseline to save memory
import gc
del baseline_model
gc.collect()
torch.cuda.empty_cache()

# 2. Load Fine-Tuned LoRA Model
print("\nLoading Fine-Tuned LoRA Model...")
ft_base = WhisperForConditionalGeneration.from_pretrained(config.base_model_id).to(device)
ft_model = PeftModel.from_pretrained(ft_base, config.lora_model_path)
if device.type == 'cuda': ft_model = ft_model.half()
ft_model.eval()

# Run Fine-Tuned Unconstrained
ft_uncon_preds, _ = run_inference(ft_model, processor, test_df, force_nepali=False)
ft_uncon_metrics = evaluate_predictions(refs, ft_uncon_preds)

# Run Fine-Tuned Constrained
ft_con_preds, _ = run_inference(ft_model, processor, test_df, force_nepali=True)
ft_con_metrics = evaluate_predictions(refs, ft_con_preds)


## Verify Byte-Identical Claim

In [ ]:
# Verify Ablation Claim: "Constrained and unconstrained decoding yield byte-identical outputs"
identical_count = sum(1 for u, c in zip(ft_uncon_preds, ft_con_preds) if u == c)
match_percentage = (identical_count / len(refs)) * 100

print("="*60)
print(f"ABLATION RESULT: {match_percentage:.2f}% byte-identical outputs on test set")
print(f"({identical_count} out of {len(refs)} utterances match perfectly)")
print("="*60)


## Table 4 Results

In [ ]:
# Generate Table 4 Data
results_table = [
    {"Model": "Whisper Zero-Shot (Constrained)", **base_metrics},
    {"Model": "Whisper-CS (Unconstrained)", **ft_uncon_metrics},
    {"Model": "Whisper-CS (Constrained)", **ft_con_metrics},
]

df_res = pd.DataFrame(results_table).round(2)
print("\nTable 4: Evaluation results on the 150-utterance test set (%)")
display(df_res)


## Paper Metrics: Hallucination, SDI Breakdown, and CMI Correlation

In [ ]:
# Section 5.1: Hallucination Rate
# Count utterances where hypothesis is >2x reference length
hallucination_count = 0
for ref, hyp in zip(refs, ft_uncon_preds):
    ref_len = len(ref.split())
    hyp_len = len(hyp.split())
    if ref_len > 0 and hyp_len > 2 * ref_len:
        hallucination_count += 1

print(f"Hallucination Rate: {hallucination_count} out of {len(refs)} utterances ({(hallucination_count/len(refs))*100:.2f}%)")


In [ ]:
# Section 5.4: SDI Error Analysis
total_s, total_d, total_i = 0, 0, 0
for ref, hyp in zip(refs, ft_uncon_preds):
    try:
        out = jiwer.process_words(ref, hyp)
        total_s += out.substitutions
        total_d += out.deletions
        total_i += out.insertions
    except:
        pass

total_errors = total_s + total_d + total_i
if total_errors > 0:
    print(f"Total Errors: {total_errors}")
    print(f"Substitutions: {total_s} ({(total_s/total_errors)*100:.1f}%)")
    print(f"Insertions: {total_i} ({(total_i/total_errors)*100:.1f}%)")
    print(f"Deletions: {total_d} ({(total_d/total_errors)*100:.1f}%)")


In [ ]:
# Section 5.5: CMI vs WER Correlation
from scipy import stats

def compute_cmi(sentence):
    tokens = sentence.split()
    ne_count = sum(1 for t in tokens if is_nepali(t))
    en_count = sum(1 for t in tokens if is_english(t))
    total = ne_count + en_count
    if total == 0: return 0
    return 100 * (1 - max(ne_count, en_count) / total)

cmis, wers = [], []
for ref, hyp in zip(refs, ft_uncon_preds):
    if not ref.strip(): continue
    cmi = compute_cmi(ref)
    if cmi > 0:
        cmis.append(cmi)
        wers.append(jiwer.wer(ref, hyp) * 100)

if len(cmis) > 2:
    r, p = stats.pearsonr(cmis, wers)
    print(f"CMI vs WER Pearson Correlation: r = {r:.3f}, p = {p:.3f}")
else:
    print("Not enough code-mixed sentences to compute correlation.")
